# code for formatting ANVIL-generated episode annotations

### imports

In [1]:
import numpy as np
import pandas as pd

### paths

In [2]:
raw_dir = '../../../data/stimuli-annotations/'
formatted_dir = '../../../data/annotations_dfs/'

### functions

In [3]:
def format_annotations(rawpath):
    """
    formats ANVIL-generated annotations into pandas dataframes
    """
    old_cols = ['Frame', 'Time', 'Scene info:narrative details - external', 
                'Scene info:narrative details - internal', 'Scene info:characters on screen', 
                'Scene info:Music Presence', 'speech:transcription', 'speech:character speaking', 
                'setting:indoor/outdoor', 'setting:setting', 'Scene name:scene name']
    new_cols = ['Frame', 'Onset time', 'Narrative details (external events)', 'Narrative details (internal state)', 
                 'Characters on screen', 'Music presence', 'Speech', 'Character speaking', 'Indoor/outdoor', 
                 'Setting', 'Scene name']
    dtypes = {col : dtype for col, dtype in 
              zip(old_cols, [np.int64, np.float64, str, str, str, np.int64, str, str, np.int64, str, str])
             }
    replace_values = {'Narrative details (external events)' : {'0' : np.nan},
                      'Narrative details (internal state)' : {'0' : np.nan},
                      'Characters on screen' : {'0' : np.nan},
                      'Music presence' : {0 : 'no', 1 : 'yes'},
                      'Speech' : {'0' : np.nan, '-1000' : np.nan},
                      'Character speaking' : {'0' : np.nan, '-1000' : np.nan},
                      'Indoor/outdoor' : {1 : 'indoor', 2 : 'outdoor', 0 : np.nan, -1000 : np.nan},
                      'Setting' : {'0' : np.nan, '-1000' : np.nan},
                      'Scene name' : {'-1000' : np.nan}
                     }
    
    df = pd.read_csv(rawpath, sep='\t', usecols=old_cols, dtype=dtypes)
    df.columns = new_cols

    # drop consecutive duplicate annotations, keep first frame only 
    # (account occasion 2-6 frame delay between start of Narrative details and Speech annotation blocks)
    df = df.loc[(df['Narrative details (external events)'].shift() != df['Narrative details (external events)']) |
               (df['Speech'].shift() != df['Speech'])]
    df = df.loc[df['Narrative details (external events)'].shift(-1) != df['Narrative details (external events)']]
    
    df.replace(replace_values, inplace=True)

    df.reset_index(drop=True, inplace=True)
    
    return df

### load annotations, format, and save out

In [4]:
for ep in ['atlep1', 'atlep2', 'arrdev']:
    rawpath = f'{raw_dir}{ep}-finished.txt'
    annot = format_annotations(rawpath)
    annot.to_csv(f'{formatted_dir}{ep}.csv', index=False)